# Komparasi Bank Big-3 Indonesia: BBCA, BMRI, BBRI

**Tujuan:** Membandingkan kinerja keuangan dan valuasi tiga bank terbesar Indonesia
berdasarkan rasio keuangan perbankan dan metode DCF.

**Bank yang dianalisis:**
- **BBCA** — PT Bank Central Asia Tbk
- **BMRI** — PT Bank Mandiri (Persero) Tbk
- **BBRI** — PT Bank Rakyat Indonesia (Persero) Tbk

**Metode:** Ratio analysis (ROE, NIM, NPL, CAR, P/B, P/E) + DCF valuation

**Data:** Laporan keuangan tahunan 2019–2023 (OJK/IDX)

**Disclaimer:** Analisis untuk tujuan edukasi. Bukan rekomendasi investasi.

---

## 0. Setup

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:,.2f}'.format)

BANKS = {
    'BBCA': {'ticker': 'BBCA.JK', 'color': 'steelblue',  'name': 'Bank Central Asia'},
    'BMRI': {'ticker': 'BMRI.JK', 'color': 'seagreen',   'name': 'Bank Mandiri'},
    'BBRI': {'ticker': 'BBRI.JK', 'color': 'coral',      'name': 'Bank Rakyat Indonesia'},
}
YEARS = [2019, 2020, 2021, 2022, 2023]
print('Setup selesai!')

## 1. Harga saham historis (via yfinance)

In [ ]:
# Ambil harga saham 5 tahun
price_data = {}
current_prices = {}
shares_out = {}

fallback_prices = {'BBCA': 9200, 'BMRI': 6250, 'BBRI': 4320}
fallback_shares = {'BBCA': 123_281_377_500, 'BMRI': 46_666_666_666, 'BBRI': 163_053_498_000}

for code, info in BANKS.items():
    try:
        ticker = yf.Ticker(info['ticker'])
        hist = ticker.history(period='5y')
        price_data[code] = hist['Close']
        cp = ticker.info.get('currentPrice', None)
        so = ticker.info.get('sharesOutstanding', None)
        current_prices[code] = cp if cp and not np.isnan(cp) else fallback_prices[code]
        shares_out[code]     = so if so and not np.isnan(so) else fallback_shares[code]
    except:
        current_prices[code] = fallback_prices[code]
        shares_out[code]     = fallback_shares[code]

print('=== HARGA SAHAM TERKINI ===')
for code in BANKS:
    mktcap = current_prices[code] * shares_out[code] / 1e12
    print(f'{code}: Rp {current_prices[code]:,}/saham | Market Cap: Rp {mktcap:.1f} Triliun')

In [ ]:
# Visualisasi perbandingan harga saham (normalized = 100 di awal)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Perbandingan Harga Saham BBCA vs BMRI vs BBRI', fontsize=14, fontweight='bold')

for code, info in BANKS.items():
    if code in price_data and len(price_data[code]) > 0:
        # Harga absolut
        axes[0].plot(price_data[code].index, price_data[code],
                    color=info['color'], linewidth=1.8, label=code)
        # Normalized (rebased ke 100)
        normalized = price_data[code] / price_data[code].iloc[0] * 100
        axes[1].plot(normalized.index, normalized,
                    color=info['color'], linewidth=1.8, label=code)

axes[0].set_title('Harga Absolut (Rp)')
axes[0].set_ylabel('Harga (Rp)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,p: f'Rp {x:,.0f}'))
axes[0].legend()

axes[1].axhline(y=100, color='gray', linestyle='--', linewidth=1)
axes[1].set_title('Normalized (Base = 100 di awal periode)')
axes[1].set_ylabel('Index')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/05_bank_harga_saham.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Data keuangan — manual dari laporan tahunan IDX/OJK

Rasio perbankan berbeda dari perusahaan biasa:
- **NIM** (Net Interest Margin) = pendapatan bunga bersih / total aset produktif
- **NPL** (Non-Performing Loan) = kredit bermasalah / total kredit
- **CAR** (Capital Adequacy Ratio) = modal / ATMR — minimum 8% per regulasi OJK
- **ROE** = net income / total equity
- **LDR** (Loan to Deposit Ratio) = total kredit / total dana pihak ketiga

In [ ]:
# ================================================================
# DATA KEUANGAN BANK BIG-3 (Rp Miliar)
# Sumber: Laporan Keuangan Tahunan 2019-2023 (IDX/OJK)
# ================================================================

data = {
    'BBCA': {
        'total_assets':    [918989, 1075570, 1228946, 1318246, 1408418],
        'total_equity':    [160121,  183979,  209788,  238944,  268443],
        'total_loans':     [566195,  560534,  617400,  719467,  818791],
        'total_deposits':  [712664,  849490,  975553, 1038710, 1106079],
        'net_income':      [ 28569,   27147,   31476,   36834,   41238],
        'net_interest_inc':[ 55406,   57431,   60622,   67111,   77502],
        'nim':             [   6.2,     5.7,     5.4,     5.5,     5.9],  # %
        'npl_gross':       [   1.3,     1.8,     2.0,     1.7,     1.9],  # %
        'car':             [  23.8,    25.8,    25.7,    25.3,    29.0],  # %
        'ldr':             [  79.4,    66.0,    63.3,    69.3,    74.1],  # %
        'shares':          [123281, 123281, 123281, 123281, 123281],  # juta lembar
        'eps':             [   232,    220,    255,    299,    334],   # Rp
        'bvps':            [  1299,   1493,   1702,   1939,   2178],  # Rp (book value per share)
    },
    'BMRI': {
        'total_assets':    [1318246, 1429334, 1598899, 1796276, 2036463],
        'total_equity':    [ 209003,  226984,  254119,  293044,  338872],
        'total_loans':     [ 913861,  908218,  988548, 1152285, 1358873],
        'total_deposits':  [ 893641,  985767, 1096982, 1200537, 1358480],
        'net_income':      [  28455,   17119,   28025,   41169,   55062],
        'net_interest_inc':[ 63489,   66360,   71432,   79547,   96823],
        'nim':             [   5.5,     4.8,     5.0,     5.1,     5.4],
        'npl_gross':       [   2.4,     3.3,     3.0,     2.4,     2.2],
        'car':             [  21.4,    19.9,    20.7,    21.4,    21.5],
        'ldr':             [ 102.2,    92.1,    90.1,    95.9,   100.0],
        'shares':          [ 46667,   46667,   46667,   46667,   46667],
        'eps':             [   610,    367,    600,    882,   1180],
        'bvps':            [  4479,   4864,   5446,   6279,   7262],
    },
    'BBRI': {
        'total_assets':    [1416759, 1511804, 1643508, 1865642, 1965000],
        'total_equity':    [ 186188,  199131,  216587,  253066,  283474],
        'total_loans':     [ 954000, 1013753, 1059470, 1242000, 1296000],
        'total_deposits':  [1018611, 1123383, 1171866, 1354000, 1431000],
        'net_income':      [  34413,   18658,   25653,   51171,   60463],
        'net_interest_inc':[ 93124,   96308,  101471,  121453,  139682],
        'nim':             [   7.4,     6.8,     6.6,     7.1,     7.9],
        'npl_gross':       [   2.6,     3.0,     3.1,     2.9,     2.8],
        'car':             [  22.6,    21.2,    22.0,    23.7,    27.5],
        'ldr':             [  93.6,    90.2,    90.4,    91.7,    90.6],
        'shares':          [163053,  163053,  163053,  163053,  163053],
        'eps':             [   211,    114,    157,    314,    371],
        'bvps':            [  1141,   1221,   1328,   1552,   1738],
    }
}

# Konversi ke DataFrame
dfs = {}
for bank, d in data.items():
    df = pd.DataFrame(d, index=YEARS)
    df['roe']       = df['net_income'] / df['total_equity'] * 100
    df['roa']       = df['net_income'] / df['total_assets'] * 100
    df['pb_ratio']  = current_prices[bank] / df['bvps']
    df['pe_ratio']  = current_prices[bank] / df['eps']
    dfs[bank] = df

print('Data keuangan berhasil dimuat!')
print('\nContoh data BBCA:')
print(dfs['BBCA'][['total_assets','net_income','roe','nim','npl_gross','car']].round(1))

## 3. Analisis rasio keuangan perbankan

In [ ]:
# Tabel komparasi rasio 2023 (tahun terbaru)
comparison_2023 = pd.DataFrame({
    bank: {
        'Total Aset (Rp T)':     dfs[bank].loc[2023, 'total_assets'] / 1000,
        'Net Income (Rp T)':     dfs[bank].loc[2023, 'net_income'] / 1000,
        'ROE (%)':               dfs[bank].loc[2023, 'roe'],
        'ROA (%)':               dfs[bank].loc[2023, 'roa'],
        'NIM (%)':               dfs[bank].loc[2023, 'nim'],
        'NPL Gross (%)':         dfs[bank].loc[2023, 'npl_gross'],
        'CAR (%)':               dfs[bank].loc[2023, 'car'],
        'LDR (%)':               dfs[bank].loc[2023, 'ldr'],
        'P/B Ratio (x)':         dfs[bank].loc[2023, 'pb_ratio'],
        'P/E Ratio (x)':         dfs[bank].loc[2023, 'pe_ratio'],
    }
    for bank in BANKS
}).T

print('=== KOMPARASI RASIO KEUANGAN 2023 ===')
print(comparison_2023.round(2).to_string())

In [ ]:
# Visualisasi perbandingan rasio
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Komparasi Rasio Keuangan BBCA vs BMRI vs BBRI (2019–2023)',
             fontsize=14, fontweight='bold')

metrics = [
    ('roe',       'ROE (%)',              axes[0,0]),
    ('nim',       'NIM (%)',              axes[0,1]),
    ('npl_gross', 'NPL Gross (%) ↓ lebih baik', axes[0,2]),
    ('car',       'CAR (%) ↑ lebih baik', axes[1,0]),
    ('ldr',       'LDR (%)',              axes[1,1]),
    ('roa',       'ROA (%)',              axes[1,2]),
]

for metric, label, ax in metrics:
    for bank, info in BANKS.items():
        ax.plot(YEARS, dfs[bank][metric],
               color=info['color'], marker='o', linewidth=2,
               markersize=6, label=bank)
    ax.set_title(label)
    ax.legend(fontsize=8)
    ax.set_xticks(YEARS)

# NPL: garis batas wajar 5%
axes[0,2].axhline(y=5, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Batas 5%')
# CAR: garis minimum OJK 8%
axes[1,0].axhline(y=8, color='red', linestyle='--', linewidth=1, alpha=0.5, label='Min OJK 8%')

plt.tight_layout()
plt.savefig('../outputs/figures/05_bank_rasio_keuangan.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Scorecard — ranking tiap rasio

In [ ]:
# Scoring: 3 = terbaik, 1 = terburuk untuk tiap rasio di 2023
metrics_scoring = {
    'ROE':  {'vals': {b: dfs[b].loc[2023,'roe']       for b in BANKS}, 'higher_better': True},
    'ROA':  {'vals': {b: dfs[b].loc[2023,'roa']       for b in BANKS}, 'higher_better': True},
    'NIM':  {'vals': {b: dfs[b].loc[2023,'nim']       for b in BANKS}, 'higher_better': True},
    'NPL':  {'vals': {b: dfs[b].loc[2023,'npl_gross'] for b in BANKS}, 'higher_better': False},
    'CAR':  {'vals': {b: dfs[b].loc[2023,'car']       for b in BANKS}, 'higher_better': True},
    'P/B':  {'vals': {b: dfs[b].loc[2023,'pb_ratio']  for b in BANKS}, 'higher_better': False},
    'P/E':  {'vals': {b: dfs[b].loc[2023,'pe_ratio']  for b in BANKS}, 'higher_better': False},
}

scorecard = pd.DataFrame(index=BANKS.keys())
for metric, info in metrics_scoring.items():
    vals = info['vals']
    sorted_banks = sorted(vals, key=lambda b: vals[b], reverse=info['higher_better'])
    scores = {sorted_banks[0]: 3, sorted_banks[1]: 2, sorted_banks[2]: 1}
    scorecard[metric] = pd.Series(scores)
    scorecard[f'{metric}_val'] = pd.Series({b: f'{vals[b]:.1f}' for b in BANKS})

score_cols = [c for c in scorecard.columns if '_val' not in c]
scorecard['TOTAL'] = scorecard[score_cols].sum(axis=1)
scorecard['RANK']  = scorecard['TOTAL'].rank(ascending=False).astype(int)

print('=== SCORECARD BANK BIG-3 (2023) ===')
print('Skor: 3=Terbaik, 2=Kedua, 1=Terburuk di tiap metrik')
print()
display_cols = score_cols + ['TOTAL', 'RANK']
print(scorecard[display_cols].to_string())
print()
winner = scorecard['TOTAL'].idxmax()
print(f'→ Bank dengan skor tertinggi: {winner} ({BANKS[winner]["name"]})')

## 5. DCF Valuation — ketiga bank

Untuk bank, DCF menggunakan **Dividend Discount Model (DDM)** atau
**Excess Return Model** karena free cash flow bank sulit diestimasi langsung.

Kita pakai pendekatan **Price-to-Book Value** yang disesuaikan dengan
justified P/B berdasarkan ROE dan cost of equity — ini standar industri
untuk valuasi bank.

In [ ]:
# === ASUMSI VALUASI ===
RISK_FREE_RATE = 0.067   # yield SBN 10Y ~6.7%
MARKET_PREMIUM = 0.055   # equity risk premium Indonesia
TERMINAL_GROWTH = 0.05   # 5% — sejalan pertumbuhan ekonomi Indonesia

# Beta tiap bank (estimasi dari volatilitas historis)
BETAS = {'BBCA': 0.70, 'BMRI': 0.85, 'BBRI': 0.90}

# Payout ratio (dividen / net income) rata-rata 3 tahun terakhir
PAYOUT = {'BBCA': 0.55, 'BMRI': 0.60, 'BBRI': 0.85}

def bank_valuation(bank_code, df, current_price, shares_million):
    """Valuasi bank dengan Justified P/B dan DDM."""
    beta   = BETAS[bank_code]
    payout = PAYOUT[bank_code]

    # Cost of equity (CAPM)
    ke = RISK_FREE_RATE + beta * MARKET_PREMIUM

    # ROE rata-rata 3 tahun terakhir
    roe_avg = df['roe'].tail(3).mean() / 100

    # Retention ratio & growth
    retention = 1 - payout
    g = roe_avg * retention   # sustainable growth rate
    g = min(g, TERMINAL_GROWTH)  # cap at terminal growth

    # Justified P/B = (ROE - g) / (ke - g)
    justified_pb = (roe_avg - g) / (ke - g)

    # Book value per share 2023
    bvps_2023 = df.loc[2023, 'bvps']

    # Intrinsic value
    intrinsic_value = justified_pb * bvps_2023

    # DDM cross-check
    eps_2023 = df.loc[2023, 'eps']
    dps      = eps_2023 * payout
    ddm_value = dps * (1 + g) / (ke - g) if ke > g else np.nan

    return {
        'cost_of_equity':   ke,
        'roe_avg':          roe_avg,
        'sustainable_g':    g,
        'justified_pb':     justified_pb,
        'bvps_2023':        bvps_2023,
        'intrinsic_value':  intrinsic_value,
        'ddm_value':        ddm_value,
        'current_price':    current_price,
        'upside':           (intrinsic_value - current_price) / current_price * 100,
    }

# Jalankan valuasi
val_results = {}
for bank in BANKS:
    val_results[bank] = bank_valuation(
        bank, dfs[bank], current_prices[bank], data[bank]['shares'][-1]
    )

# Tampilkan hasil
print('=== HASIL VALUASI BANK BIG-3 ===')
print(f'{"":28} {"BBCA":>10} {"BMRI":>10} {"BBRI":>10}')
print('-' * 62)
rows = [
    ('Cost of Equity',     lambda b: f'{val_results[b]["cost_of_equity"]:>10.1%}'),
    ('ROE Avg 3Y',         lambda b: f'{val_results[b]["roe_avg"]:>10.1%}'),
    ('Sustainable Growth', lambda b: f'{val_results[b]["sustainable_g"]:>10.1%}'),
    ('Justified P/B',      lambda b: f'{val_results[b]["justified_pb"]:>10.2f}x'),
    ('BVPS 2023 (Rp)',     lambda b: f'{val_results[b]["bvps_2023"]:>10,}'),
    ('Intrinsic Value (Rp)',lambda b: f'{val_results[b]["intrinsic_value"]:>10,.0f}'),
    ('DDM Value (Rp)',     lambda b: f'{val_results[b]["ddm_value"]:>10,.0f}'),
    ('Harga Pasar (Rp)',   lambda b: f'{val_results[b]["current_price"]:>10,}'),
    ('Upside/Downside',    lambda b: f'{val_results[b]["upside"]:>+10.1f}%'),
]
for label, fn in rows:
    print(f'{label:28}' + ''.join(fn(b) for b in BANKS))

print()
print('=== VERDICT ===')
for bank in BANKS:
    iv  = val_results[bank]['intrinsic_value']
    cp  = val_results[bank]['current_price']
    pct = val_results[bank]['upside']
    verdict = 'UNDERVALUED ✓' if iv > cp else 'OVERVALUED ✗'
    print(f'{bank}: Rp {iv:>6,.0f} vs Rp {cp:,}  {pct:>+6.1f}%  →  {verdict}')

## 6. Visualisasi hasil valuasi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Valuasi Bank Big-3 Indonesia', fontsize=14, fontweight='bold')

banks = list(BANKS.keys())
colors = [BANKS[b]['color'] for b in banks]

# Plot 1: Intrinsic value vs harga pasar
iv_vals = [val_results[b]['intrinsic_value'] for b in banks]
cp_vals = [val_results[b]['current_price']   for b in banks]

x = np.arange(len(banks))
width = 0.35
bars1 = axes[0].bar(x - width/2, iv_vals, width, color=colors, alpha=0.85, label='Intrinsic Value')
bars2 = axes[0].bar(x + width/2, cp_vals, width, color='gray',  alpha=0.5,  label='Harga Pasar')
axes[0].set_xticks(x)
axes[0].set_xticklabels(banks, fontsize=12)
axes[0].set_title('Intrinsic Value vs Harga Pasar (Rp)')
axes[0].set_ylabel('Harga (Rp)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,p: f'Rp {x:,.0f}'))
axes[0].legend()
for bar, val in zip(bars1, iv_vals):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
                f'Rp {val:,.0f}', ha='center', fontsize=8, fontweight='bold')

# Plot 2: Upside/downside
upsides = [val_results[b]['upside'] for b in banks]
bar_colors = ['seagreen' if u > 0 else 'coral' for u in upsides]
bars3 = axes[1].bar(banks, upsides, color=bar_colors, alpha=0.85, width=0.5)
axes[1].axhline(y=0, color='black', linewidth=1)
axes[1].set_title('Upside / Downside vs Harga Pasar (%)')
axes[1].set_ylabel('%')
for bar, val in zip(bars3, upsides):
    offset = 0.5 if val >= 0 else -1.5
    axes[1].text(bar.get_x()+bar.get_width()/2, val+offset,
                f'{val:+.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/05_bank_valuasi.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Radar chart — profil keseluruhan

In [ ]:
# Normalize semua metrik ke skala 0-1 untuk radar chart
radar_metrics = {
    'ROE':  {b: dfs[b].loc[2023,'roe']       for b in banks},
    'NIM':  {b: dfs[b].loc[2023,'nim']       for b in banks},
    'CAR':  {b: dfs[b].loc[2023,'car']       for b in banks},
    'NPL\n(inv)': {b: 10 - dfs[b].loc[2023,'npl_gross'] for b in banks},  # inverse: lower NPL = better
    'ROA':  {b: dfs[b].loc[2023,'roa']       for b in banks},
    'Upside': {b: max(0, val_results[b]['upside']) for b in banks},
}

# Normalize ke 0-1
norm = {}
for metric, vals in radar_metrics.items():
    mn, mx = min(vals.values()), max(vals.values())
    norm[metric] = {b: (v-mn)/(mx-mn) if mx > mn else 0.5 for b, v in vals.items()}

labels   = list(radar_metrics.keys())
N        = len(labels)
angles   = [n / float(N) * 2 * np.pi for n in range(N)]
angles  += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
fig.suptitle('Profil Bank Big-3 — Radar Chart (2023)', fontsize=13, fontweight='bold')

for bank in banks:
    values = [norm[m][bank] for m in labels] + [norm[labels[0]][bank]]
    ax.plot(angles, values, color=BANKS[bank]['color'], linewidth=2, label=bank)
    ax.fill(angles, values, color=BANKS[bank]['color'], alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=11)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['25%','50%','75%','100%'], fontsize=7)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig('../outputs/figures/05_bank_radar.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Kesimpulan & Rekomendasi

### Hasil Valuasi

| Bank | Intrinsic Value | Harga Pasar | Upside/Downside | Verdict |
|------|----------------|-------------|-----------------|---------|
| BBCA | Rp 4.026 | Rp 5.700 | -29.4% | OVERVALUED ✗ |
| BMRI | Rp 9.996 | Rp 4.080 | +145.0% | UNDERVALUED ✓ |
| BBRI | Rp 2.928 | Rp 2.950 | -0.8% | OVERVALUED ✗ |

---

### Temuan Utama

**BMRI — Most Undervalued, Strongest Momentum**

Bank Mandiri adalah temuan paling menarik dari analisis ini. Dengan
intrinsic value Rp 9.996 vs harga pasar Rp 4.080, BMRI menunjukkan
upside potensial sebesar **+145%** — angka yang signifikan. Ini
didorong oleh kombinasi ROE rata-rata 3 tahun yang kuat (13.8%),
justified P/B 1.38x yang jauh di atas actual P/B saat ini, dan
pertumbuhan net income yang paling agresif di antara tiga bank
(dari Rp 17.1T di 2020 ke Rp 55.1T di 2023 — naik 3x dalam 3 tahun).

**BBCA — Premium Quality, Premium Price**

BBCA mempertahankan posisinya sebagai bank berkualitas tertinggi
secara fundamental: NPL terendah (1.9%), CAR tertinggi (29%), dan
NIM yang stabil. Namun kualitas ini sudah sepenuhnya tercermin di
harga pasar — bahkan lebih. Dengan intrinsic value Rp 4.026 vs
harga pasar Rp 5.700, BBCA terlihat *overvalued* 29.4% berdasarkan
model justified P/B. Investor yang memegang BBCA membayar premium
untuk stabilitas dan brand trust — sebuah trade-off yang valid
untuk investor defensif, namun kurang menarik untuk value investor.

**BBRI — Fairly Valued dengan Upside Terbatas**

BBRI berada di posisi tengah — hampir perfectly fairly valued
(-0.8% dari intrinsic value). NIM tertinggi di antara ketiganya
(7.9%) mencerminkan keunggulan di segmen mikro dan UMKM, namun
ROE yang lebih rendah dari BBCA membuat justified P/B-nya lebih
konservatif. BBRI cocok untuk investor yang mencari eksposur ke
pertumbuhan ekonomi grass-root Indonesia.

---

### Trade-off Kualitas vs Valuasi

Analisis ini mengungkapkan trade-off klasik antara kualitas dan valuasi:

- **BBCA:** kualitas tertinggi → harga paling mahal → upside terbatas
- **BMRI:** kualitas menengah → harga paling murah → upside terbesar
- **BBRI:** kualitas menengah-tinggi → harga fairly valued → upside minimal

Ini konsisten dengan teori pasar efisien — kualitas yang superior
sudah di-price-in oleh pasar. Nilai tersembunyi justru lebih sering
ditemukan di bank yang sedang dalam fase transformasi seperti BMRI.

---

### Rekomendasi (horizon 3–5 tahun)

| Bank | Skor Kualitas | Intrinsic Value | vs Pasar | Rekomendasi |
|------|--------------|----------------|----------|-------------|
| BBCA | Tertinggi | Rp 4.026 | -29.4% | HOLD — jangan beli di harga ini |
| BMRI | Menengah | Rp 9.996 | +145.0% | BUY — undervalued signifikan |
| BBRI | Menengah | Rp 2.928 | -0.8% | HOLD — fairly valued |

**Pick terbaik: BMRI** — kombinasi momentum earnings yang kuat,
valuasi yang masih murah, dan posisi sebagai bank BUMN terbesar
menjadikannya pilihan paling menarik secara risk-adjusted.

---

### Keterbatasan Model

- Justified P/B sensitif terhadap asumsi cost of equity dan
  sustainable growth rate
- Model tidak memperhitungkan risiko regulasi OJK atau perubahan
  kebijakan dividen
- Data 2020 yang terpengaruh COVID dapat mendistorsi rata-rata ROE
- Bank BUMN (BMRI, BBRI) memiliki risiko intervensi pemerintah
  yang tidak tertangkap oleh model kuantitatif

---

*Disclaimer: Analisis ini dibuat untuk tujuan edukasi dan portofolio
akademik. Bukan merupakan rekomendasi investasi.*

*Analisis oleh: Armandya Danu | Ekonomi Universitas Brawijaya | 31 Mei 2026*